# Droplet rebound simulation - Initial conditions

This notebook is used for computing initial conditions for the velocity field of the droplet rebound test case. Since the droplet has an impact velocity (VelocityZ#A), we increase the corresponding field successivley form 0 to the desired impact velocity value.
The resulting solution will be stored in designated database, from which the actual simulation, see `DropletRebound.ipynb`, will be restarted.

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Databases loaded: 
Capacity: 0
Count: 0



Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 204
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 97
   at Submission#26.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;

In [ ]:
BoSSSshell.WorkflowMgm.Init("DropletReboundInit");

Project name is set to 'DropletReboundInit'.
Default Execution queue is chosen for the database.
Opening existing database 'D:\local\DropletReboundInit'.


In [ ]:
//ExecutionQueues

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,\\hpccluster\hpccluster-scratch\smuda\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,DC2,<null>,Normal,True,[ \\hpccluster\hpccluster-scratch\smuda\ ]


## Grid creation

The used grid is a cartesain box around the droplet injection location, which is located at `radiusOP` (in $r$-direction) away from the center of the rotating disk.

In [ ]:
static public Grid3D RotatingDiskSector_CartesianCutOut(double radiusOP, double l_radial, double l_azimuthal, double h_axial, int res_radial, int res_azimuthal, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(radiusOP - (l_radial / 2.0), radiusOP + (l_radial / 2.0), res_radial + 1);    // radial direction
    double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 2.0, l_azimuthal / 2.0, res_azimuthal + 1);    // azimuthal direction
    // double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 3.0, (2.0 * l_azimuthal) / 3.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_CartesianCutOut_{res_radial}x{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "velocity_inlet_rotatingDisk");
    // grd.EdgeTagNames.Add(2, "velocity_inlet_top");
    grd.EdgeTagNames.Add(2, "pressure_outlet_top");
    grd.EdgeTagNames.Add(3, "velocity_inlet_back");
    // grd.EdgeTagNames.Add(4, "velocity_inlet_front");
    grd.EdgeTagNames.Add(4, "pressure_outlet_front");
    grd.EdgeTagNames.Add(5, "velocity_inlet_upstream");
    grd.EdgeTagNames.Add(6, "pressure_outlet_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.z.Abs() <= 1e-8)
            et = 1;
        if ((X.z - h_axial).Abs() <= 1e-8)
            et = 2;
        if (((X.x - radiusOP) + (l_radial / 2.0)).Abs() <= 1e-8)
            et = 3;
        if (((X.x - radiusOP) - (l_radial / 2.0)).Abs() <= 1e-8)
            et = 4;
        if ((X.y + (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 5;
        if ((X.y - (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 6;
        // if ((X.y + (l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 5;
        // if ((X.y - (2.0 * l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 6;

        return et;
    });    

    return grd;
}

## simulation setup

In [ ]:
double radiusOP = 78e-3; // operating point (droplet injection point) -> Re = radiusOp / Lstar
double density = 1.19;  
double kinViscosity = 1.52e-5; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double omega = 1820.5128205128206; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(kinViscosity / omega);  // used for non-dimensionalization of the flow fields
Lstar

9.137448098155134E-05

Reynolds number at the point of injection

In [ ]:
double reynolds = radiusOP / Lstar;
reynolds

853.6300196960717

Boundary layer (BL) thickness

In [ ]:
double zBL = 5.5;   // dimesionless value
double zBLstar = zBL * Lstar;
zBLstar

0.0005025596453985323

Height of the computational domain (roughly 3 times the BL thickness)

In [ ]:
double zTop = 15;
double zTopStar = zTop * Lstar;
zTopStar

0.00137061721472327

## Prescribed boundary conditions (initial conditions) for the flow field 

The B.C. (respectively I.C. also) are given by the Homotopy Analysis method (HAM), which give a linear combination in term of $\sum_{n,i} \textrm{e}^{-n \eta} \eta^{i}$, where $\eta$ describes the dimensionless height in axial-direction.

In [ ]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [ ]:
MultidimensionalArray HAMcoeff_velU = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffU.txt");
MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffV.txt");
MultidimensionalArray HAMcoeff_velW = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffW.txt");
MultidimensionalArray HAMcoeff_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffP.txt");

In [ ]:
var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityX();
vonKarmanHAM_velX.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity, density);
var vonKarmanHAM_velY = new RotatingDiskBoundaryLayerConditionsHAM_VelocityY();
vonKarmanHAM_velY.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity, density);
var vonKarmanHAM_velZ = new RotatingDiskBoundaryLayerConditionsHAM_VelocityZ();
vonKarmanHAM_velZ.SetData(HAMcoeff_velW, omega, kinViscosity, density);
var vonKarmanHAM_P = new RotatingDiskBoundaryLayerConditionsHAM_PressureP();
vonKarmanHAM_P.SetData(HAMcoeff_P, omega, kinViscosity, density);

### B.C. for the rotating disk

One may also use the B.C. for the analytic solution

In [ ]:
Formula RotatingDiskVelocityX = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double omega = 1820.5128205128206; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velX = - rOnRotDisk * Math.Sin(theta) * omega;" +
    "return velX; } "
);

In [ ]:
Formula RotatingDiskVelocityY = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double omega = 1820.5128205128206; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velY = rOnRotDisk * Math.Cos(theta) * omega;" +
    "return velY; } "
);

### B.C. for the top boundary 

Again, one may use the B.C. for the analytic solution

In [ ]:
double velW_top = -0.88447342;
double velWstar_top = velW_top * Math.Sqrt(kinViscosity * omega); 
velWstar_top

-0.14713075072584397

In [ ]:
Formula VelocityZ_top = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.14713075072584397; } "
);

## Initial conditions for the injected droplet 

In [ ]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { " +
    "double radiusOP = 78e-3;" +
    "double radiusDrop = 0.205e-3;" +
    "double initHeight = 75e-5;" +
    "return Math.Sqrt((X[0] - radiusOP).Pow2() + X[1].Pow2() + (X[2] - initHeight).Pow2()) - radiusDrop; } "
);

In [ ]:
// Formula InitVelocity = new Formula(
//     "VelZ",
//     false,
//     "double VelZ(double[] X) { return -1.12; } "
// );

Instead of setting the initial velocity of the droplet directly, we employ a ramp up from 0 to the desired value. 

In [ ]:
double[] rampUpValues_DropVelocityZ = new double[] { 0.0, -0.112, -0.56, -1.12 };

## Control settings

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
var C = new XNSE_Control();

int k = 3;
C.SetDGdegree(k);


// physical parameters
double densityDrop = 998.37;
C.PhysicalParameters.rho_A = densityDrop;
C.PhysicalParameters.mu_A = densityDrop * 0.000001028;

C.PhysicalParameters.rho_B = density;
C.PhysicalParameters.mu_B = density * kinViscosity;

C.PhysicalParameters.Sigma = 72.9e-3;

C.PhysicalParameters.IncludeConvection = true;

    
// set grid
double l_radial = zTopStar; 
double l_azimuthal = zTopStar;
// double l_azimuthal = (3.0/2.0) * zTopStar;
double h_axial = zTopStar; 

int res_global = 8;
int res_radial = 1 * res_global; 
int res_azimuthal = 1 * res_global;
// int res_azimuthal = (3 * res_global) / 2;
int res_axial = 1 * res_global;

Grid3D grd = RotatingDiskSector_CartesianCutOut(radiusOP, l_radial, l_azimuthal, h_axial, res_radial, res_azimuthal, res_axial);
C.SetGrid(grd);

// boundary conditions
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityX#B", RotatingDiskVelocityX);
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityY#B", RotatingDiskVelocityY);

// C.AddBoundaryValue("velocity_inlet_top", "VelocityX#B", vonKarmanHAM_velX);
// C.AddBoundaryValue("velocity_inlet_top", "VelocityY#B", vonKarmanHAM_velY);
// C.AddBoundaryValue("velocity_inlet_top", "VelocityZ#B", vonKarmanHAM_velZ);

// C.AddBoundaryValue("velocity_inlet_front", "VelocityX#B", vonKarmanHAM_velX);
// C.AddBoundaryValue("velocity_inlet_front", "VelocityY#B", vonKarmanHAM_velY);
// C.AddBoundaryValue("velocity_inlet_front", "VelocityZ#B", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_back", "VelocityX#B", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_back", "VelocityY#B", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_back", "VelocityZ#B", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#B", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityY#B", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityZ#B", vonKarmanHAM_velZ);

// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#B", vonKarman_velX);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityY#B", vonKarman_velY);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityZ#B", vonKarman_velZ);
// C.AddBoundaryValue("pressure_dirichlet_downstream", "Pressure#B", vonKarman_P);


// initial conditions
C.AddInitialValue("VelocityX#B", vonKarmanHAM_velX);
C.AddInitialValue("VelocityY#B", vonKarmanHAM_velY);
C.AddInitialValue("VelocityZ#B", vonKarmanHAM_velZ);
// C.AddInitialValue("Pressure#B", vonKarman_P);


C.AddInitialValue("Phi", PhiFunc);
// C.AddInitialValue("VelocityZ#A", InitVelocity);

C.AddInitialRampUpValue("VelocityZ#A", new ConstantValue(0.0));
C.AddInitialRampUpValue("VelocityZ#A", new ConstantValue(-0.112));
C.AddInitialRampUpValue("VelocityZ#A", new ConstantValue(-0.56));
C.AddInitialRampUpValue("VelocityZ#A", new ConstantValue(-1.12));



C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;


// C.SkipSolveAndEvaluateResidual = true;

C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 20;

// C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


// C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
C.Timestepper_LevelSetHandling = LevelSetHandling.None;
C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
// C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
// C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.dtFixed = 1.0e-6;
C.NoOfTimesteps = 5;
    
{
    C.AdaptiveMeshRefinement = true;
    int AMRlevel = 2;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel });
    C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1, 3, 5 }) { maxRefinementLevel = AMRlevel - 1 });
    C.AMR_startUpSweeps = AMRlevel * 2;
}

// C.PostprocessingModules.Add(new EnergyLogging());
    
C.SessionName = "DropletRebound_8x8x16AMR2_Picard4_InitState";
    
Controls.Add(C);

Grid Edge Tags changed.


## Launch job

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,DropletRebound_8x8x16AMR2_Picard4_InitState


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    oneJob.NumberOfMPIProcs = 1;
    oneJob.Activate(myBatch); 
}

Deploying job DropletRebound_8x8x16AMR2_Picard4_InitState ... 
Opening existing database '\\hpccluster\hpccluster-scratch\smuda\DropletReboundInit'.
Set Database: { Session Count = 12; Grid Count = 33; Path = \\hpccluster\hpccluster-scratch\smuda\DropletReboundInit }
Grid is not in database yet...
Grid successfully saved: 258cec15-8b8a-422d-93ee-188ff63a8704
Deploying executables and additional files ...
Deployment directory: \\hpccluster\hpccluster-scratch\smuda\binaries\DropletReboundInit-XNSE_Solver2023Sep01_145426.685780
copied 42 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.



## Move restart session

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletReboundInit");
var initDB = databases.Pick(1);
initDB

Opening existing database '\\hpccluster\hpccluster-scratch\smuda\DropletReboundInit'.


{ Session Count = 4; Grid Count = 8; Path = \\hpccluster\hpccluster-scratch\smuda\DropletReboundInit }

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletRebound");
var runDB = databases.Pick(2);
runDB

Opening existing database '\\hpccluster\hpccluster-scratch\smuda\DropletRebound'.


{ Session Count = 18; Grid Count = 36; Path = \\hpccluster\hpccluster-scratch\smuda\DropletRebound }

In [ ]:
//initDB.Sessions.Pick(0).Move(runDB);

## Postprocessing

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletReboundInit");

In [ ]:
databases.Pick(1).Sessions

#0: DropletReboundInit	DropletRebound_8x8x16AMR1_Picard2_InitState	08/31/2023 13:53:22	042ec43c...
#1: DropletReboundInit	DropletRebound_8x8x16AMR1_NewtonKcycleSchwarz2_InitState*	08/31/2023 13:54:25	3c60211b...
#2: DropletReboundInit	DropletRebound_8x8x16AMR1_Newton2_InitState*	08/31/2023 13:53:56	d0941bd4...
#3: DropletReboundInit	DropletRebound_8x8x16AMR1_Picard_InitState*	08/31/2023 13:22:19	dbe42820...
#4: DropletReboundInit	DropletRebound_8x8x16AMR1_Newton_InitState*	08/31/2023 13:21:43	7d9609d9...
#5: DropletReboundInit	DropletRebound_8x8x16AMR1_NewtonKcycleSchwarz_InitState*	08/31/2023 13:20:53	ee240c69...
#6: DropletReboundInit	DropletRebound_8x8x16AMR1_InitState*	08/31/2023 13:19:05	ff67de6d...
#7: DropletReboundInit	DropletRebound-TestRunRampUpInitDropVelocity7	08/31/2023 09:16:47	5e599b03...
#8: DropletReboundInit	DropletRebound-TestRunRampUpInitDropVelocity6	08/28/2023 15:18:29	84ff05b1...
#9: DropletReboundInit	DropletRebound-TestRunRampUpInitDropVelocity5*	08/28/2023 15:

In [ ]:
var sess = databases.Pick(1).Sessions.Pick(0);
sess

DropletReboundInit	DropletRebound_8x8x16AMR1_Picard2_InitState	08/31/2023 13:53:22	042ec43c...

In [ ]:
sess.Timesteps

#0:  { Time-step: 0.0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#1:  { Time-step: 0.1; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#2:  { Time-step: 0.2; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#3:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean,

In [ ]:
sess.Timesteps.Skip(3).Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletReboundInit__DropletRebound_8x8x16AMR1_Picard2_InitState__042ec43c-950e-458c-b5be-cd1e11c25da4


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletReboundInit__DropletRebound_8x8x16AMR1_Picard2_InitState__042ec43c-950e-458c-b5be-cd1e11c25da4

### Velocity profiles

In [ ]:
int tIdx = 2;
sess.Timesteps.Pick(tIdx)

 { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }

In [ ]:
DGField[] solutionFields = new DGField[4];
solutionFields[0] = sess.Timesteps.Pick(tIdx).GetField("VelocityX");
solutionFields[1] = sess.Timesteps.Pick(tIdx).GetField("VelocityY");
solutionFields[2] = sess.Timesteps.Pick(tIdx).GetField("VelocityZ");
solutionFields[3] = sess.Timesteps.Pick(tIdx).GetField("Pressure");

In [ ]:
double[][] solutionValues = new double[4][];
double[] evalPoint = new double[] {radiusOP - 1.0e-5, 1.0e-5};

int numVal = 100;
double stepSize = zTopStar / numVal; 
double[] zStarValues = Enumerable.Range(0, numVal+1).Select(idx => idx * stepSize).ToArray();

// velocity fields
for (int j = 0; j < 3; j++) {
    solutionValues[j] = new double[numVal+1];
    for (int i = 0; i <= numVal; i++) {
        try {
            solutionValues[j][i] = solutionFields[j].ProbeAt(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]});
        }
        catch {
            Console.WriteLine("evalPoint = ( {0}, {1}, {2} ) not found", evalPoint[0], evalPoint[1], zStarValues[i]);
        }
    }
}
// pressure field (correct to same pressure level at wall p=0)
solutionValues[3] = new double[numVal+1];
double pressure0 = solutionFields[3].ProbeAt(new double[] {evalPoint[0], evalPoint[1], 0.0});
for (int i = 0; i <= numVal; i++) {
    try {
        solutionValues[3][i] = solutionFields[3].ProbeAt(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}) - pressure0;
    }
    catch {
        Console.WriteLine("evalPoint = ( {0}, {1}, {2} ) not found", evalPoint[0], evalPoint[1], zStarValues[i]);
    }
}

In [ ]:
double[][] varValues = new double[4][];
varValues[0] = new double[numVal+1];
varValues[1] = new double[numVal+1];
varValues[2] = new double[numVal+1];
varValues[3] = new double[numVal+1];

for (int i = 0; i <= numVal; i++) {
    varValues[0][i] = vonKarman_velX.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    varValues[1][i] = vonKarman_velY.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0); 
    varValues[2][i] = vonKarman_velZ.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    varValues[3][i] = vonKarman_P.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
}

In [ ]:
var fmt1 = new PlotFormat();    // blue - BoSSS
fmt1.LineColor = LineColors.Blue;
var fmt2 = new PlotFormat();    // red - Analytic

var gp = new Gnuplot();
gp.SetMultiplot(2,2);

int idx = 0;
for (int i = 0; i < 2; i++)
for (int j = 0; j < 2; j++) {
    Plot2Ddata plt = new Plot2Ddata(); 
    plt.AddDataGroup("BoSSS", solutionValues[idx], zStarValues, fmt1);
    plt.AddDataGroup("Analytic", varValues[idx], zStarValues, fmt2);
    gp.SetSubPlot(i,j, varNames[idx]);
    plt.ToGnuplot(gp);
    idx++;
}
//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:1000)

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.0001 
 
 
 
 
 0.0002 
 
 
 
 
 0.0003 
 
 
 
 
 0.0004 
 
 
 
 
 0.0005 
 
 
 
 
 0.0006 
 
 
 
 
 0.0007 
 
 
 
 
 0.0008 
 
 
 
 
 0.0009 
 
 
 
 
 0.001 
 
 
 
 
 -5 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 25 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 BoSSS 
 
 
 BoSSS 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M531.0,62.1 L584.4,62.1 M172.5,459.1 L286.2,455.3 L378.3,451.5 L451.4,447.6 L508.0,443.8 L550.5,440.0
 L580.9,436.2 L601.0,432.4 L612.7,428.5 L617.3,424.7 L615.9,420.9 L609.8,417.1 L600.0,413.3 L587.4,409.4
 L572.5,405.6 L556.2,401.8 L538.7,398.0 L520.5,394.2 L501.9,390.3 L483.4,386.5 L465.1,382.7 L447.2,378.9
 L429.7,375.1 L412.9,371.3 L396.7,367.4 L381.3,363.6 L366.6,359.8 L352.7,356.0 L339.7,352.2 L327.3,348.3
 L315.8,344.5 L305.0,340.7 L294.9,336.9 L285.4,333.1 L276.7,329.2 L268.5,325.4 L261.0,321.6 L254.0,317.8
 L247.5,314.0 L241.5,310.1 L236.0,306.3 L230.8,302.5 L226.1,298.7 L221.8,294.9 L217.8,291.0 L214.1,287.2
 L210.7,283.4 L207.6,279.6 L204.7,275.8 L202.0,271.9 L199.6,268.1 L197.4,264.3 L195.3,260.5 L193.5,256.7
 L191.7,252.8 L190.2,249.0 L188.7,245.2 L187.4,241.4 L186.2,237.6 L185.0,233.8 L184.0,229.9 L183.1,226.1
 L182.2,222.3 L181.4,218.5 L180.7,214.7 L180.0,210.8 L179.4,207.0 L178.9,203.2 L178.4,199.4 L177.9,195.6
 L177.5,191.7 L177.1,187.9 L176.7,184.1 L176.4,180.3 L176.1,176.5 L175.8,172.6 L175.6,168.8 L175.3,165.0
 L175.1,161.2 L174.9,157.4 L174.7,153.5 L174.6,149.7 L174.4,145.9 L174.3,142.1 L174.2,138.3 L174.1,134.4
 L174.0,130.6 L173.9,126.8 L173.8,123.0 L173.7,119.2 L173.6,115.3 L173.6,111.5 L173.5,107.7 L173.4,103.9
 L173.4,100.1 L173.3,96.3 L173.3,92.4 L173.3,88.6 L173.2,84.8 L173.2,81.0 L173.2,77.2 '/> 
 
 Analytic 
 
 
 Analytic 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M531.0,86.1 L584.4,86.1 M172.5,459.1 L286.2,455.3 L378.3,451.5 L451.4,447.6 L508.0,443.8 L550.5,440.0
 L580.9,436.2 L601.1,432.4 L612.7,428.5 L617.2,424.7 L615.8,420.9 L609.8,417.1 L600.1,413.3 L587.4,409.4
 L572.6,405.6 L556.1,401.8 L538.6,398.0 L520.5,394.2 L502.0,390.3 L483.5,386.5 L465.1,382.7 L447.2,378.9
 L429.7,375.1 L412.9,371.3 L396.7,367.4 L381.3,363.6 L366.6,359.8 L352.7,356.0 L339.6,352.2 L327.3,348.3
 L315.8,344.5 L305.0,340.7 L294.9,336.9 L285.4,333.1 L276.7,329.2 L268.5,325.4 L261.0,321.6 L254.0,317.8
 L247.5,314.0 L241.5,310.1 L236.0,306.3 L230.8,302.5 L226.1,298.7 L221.8,294.9 L217.8,291.0 L214.1,287.2
 L210.7,283.4 L207.6,279.6 L204.7,275.8 L202.0,271.9 L199.6,268.1 L197.4,264.3 L195.3,260.5 L193.5,256.7
 L191.7,252.8 L190.2,249.0 L188.7,245.2 L187.4,241.4 L186.2,237.6 L185.0,233.8 L184.0,229.9 L183.1,226.1
 L182.2,222.3 L181.4,218.5 L180.7,214.7 L180.0,210.8 L179.4,207.0 L178.9,203.2 L178.4,199.4 L177.9,195.6
 L177.5,191.7 L177.1,187.9 L176.7,184.1 L176.4,180.3 L176.1,176.5 L175.8,172.6 L175.6,168.8 L175.3,165.0
 L175.1,161.2 L174.9,157.4 L174.7,153.5 L174.6,149.7 L174.4,145.9 L174.3,142.1 L174.2,138.3 L174.1,134.4
 L174.0,130.6 L173.9,126.8 L173.8,123.0 L173.7,119.2 L173.6,115.3 L173.6,111.5 L173.5,107.7 L173.4,103.9
 L173.4,100.1 L173.3,96.3 L173.3,92.4 L173.3,88.6 L173.2,84.8 L173.2,81.0 L173.2,77.2 '/> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.0001 
 
 
 
 
 0.0002 
 
 
 
 
 0.0003 
 
 
 
 
 0.0004 
 
 
 
 
 0.0005 
 
 
 
 
 0.0006 
 
 
 
 
 0.0007 
 
 
 
 
 0.0008 
 
 
 
 
 0.0009 
 
 
 
 
 0.001 
 
 
 
 
 0 
 
 
 
 
 20 
 
 
 
 
 40 
 
 
 
 
 60 
 
 
 
 
 80 
 
 
 
 
 100 
 
 
 
 
 120 
 
 
 
 
 140 
 
 
 
 
 160 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 